# Assignment 1 — QANet

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Run each section in order. Sections 0–1 are one-time setup steps; Sections 2–4 are the main training and evaluation pipeline.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust this path if your repo is stored elsewhere in Drive.
PROJECT_ROOT = "/content/drive/MyDrive/Assignment1_2026"
#PROJECT_ROOT = "/Users/nathanroland/Desktop/COMP4329/a1/Assignment1_2026"

In [ ]:
# Install Python dependencies (run once per session)
!pip install -r {PROJECT_ROOT}/requirements.txt -q
!python -m spacy download en

---
## Section 0 — Environment Setup

Mount Google Drive and install dependencies.

In [ ]:
import sys, os

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

---
## Section 1 — Download Data *(delete before submitting)*

Downloads the pre-built mini dataset (sampled SQuAD v1.1 train + full dev set,
with GloVe vectors filtered to the mini vocabulary) from GitHub Releases into `_data/`.

> **One-time step.** Once `_data/` exists on your Drive, delete this section before submission.

In [ ]:
# from Tools.download import download_mini

# download_mini(data_dir="_data")

---
## Section 2 — Preprocess Data *(delete before submitting)*

Tokenises the SQuAD JSON files, builds word/char vocabularies from GloVe, and writes padded index tensors to `_data/`.

> **One-time step.** Once `_data/*.npz` exists on your Drive, delete this section before submission. Re-run only if you change `para_limit`, `ques_limit`, or other shape parameters.

In [ ]:
from Tools.preproc import preprocess

preprocess(
    train_file="_data/squad/train-mini.json",
    dev_file="_data/squad/dev-v1.1.json",
    glove_word_file="_data/glove/glove.mini.txt",
    target_dir="_data",
    para_limit=400,
    ques_limit=50,
)

---
## Section 3 — Train

Trains QANet on SQuAD v1.1 and saves the best checkpoint to `_model/model.pt`.

In [ ]:
from TrainTools.train import train

import os
import time
import json
import pandas as pd

os.makedirs("experiments/optimizer_compare", exist_ok=True)

runs = [
    {
        "optimizer_name": "sgd",
        "save_dir": "_model_sgd",
        "log_dir": "_log_sgd",
        "num_steps": 30000,
        "checkpoint": 500,
    },
    {
        "optimizer_name": "sgd_momentum",
        "save_dir": "_model_sgd_momentum",
        "log_dir": "_log_sgd_momentum",
        "num_steps": 30000,
        "checkpoint": 500,
    },
    {
        "optimizer_name": "adam",
        "save_dir": "_model_adam",
        "log_dir": "_log_adam",
        "num_steps": 30000,
        "checkpoint": 500,
    },
]

In [ ]:
summary_rows = []

for r in runs:
    t0 = time.time()

    # Pass the per-run config directly so custom hyperparameters are honored.
    results = train(**r)

    elapsed = time.time() - t0

    run_id = r["optimizer_name"]
    out_dir = f"experiments/optimizer_compare/{run_id}"
    os.makedirs(out_dir, exist_ok=True)

    # 1) Persist full history
    hist_df = pd.DataFrame(results["history"])
    hist_df.to_csv(f"{out_dir}/history.csv", index=False)
    hist_df.to_json(f"{out_dir}/history.json", orient="records", indent=2)

    # 2) Persist compact summary
    row = {
        "optimizer": r["optimizer_name"],
        "best_f1": results["best_f1"],
        "best_em": results["best_em"],
        "best_ckpt_path": results["best_ckpt_path"],
        "ckpt_path": results["ckpt_path"],
        "wall_clock_seconds": elapsed,
    }
    summary_rows.append(row)

    with open(f"{out_dir}/summary.json", "w") as f:
        json.dump(row, f, indent=2)

summary_df = pd.DataFrame(summary_rows)
summary_df

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

history_paths = {
    "sgd": "experiments/optimizer_compare/sgd/history.csv",
    "sgd_momentum": "experiments/optimizer_compare/sgd_momentum/history.csv",
    "adam": "experiments/optimizer_compare/adam/history.csv",
}

histories = {}
for optimizer_name, history_path in history_paths.items():
    if not os.path.exists(history_path):
        raise FileNotFoundError(
            f"Missing history file: {history_path}. Run the corresponding training experiment first."
        )
    histories[optimizer_name] = pd.read_csv(history_path)

summary_rows = []
for optimizer_name, history_df in histories.items():
    best_idx = history_df["dev_f1"].idxmax()
    best_row = history_df.loc[best_idx]
    summary_rows.append({
        "optimizer": optimizer_name,
        "convergence_step": int(best_row["step"]),
        "best_dev_f1": float(best_row["dev_f1"]),
        "best_dev_em": float(best_row["dev_em"]),
        "final_train_loss": float(history_df["train_loss"].iloc[-1]),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("optimizer")
summary_df

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8), sharex=True)

for optimizer_name, history_df in histories.items():
    # Top row: train metrics
    axes[0, 0].plot(history_df["step"], history_df["train_loss"], label=optimizer_name)
    axes[0, 1].plot(history_df["step"], history_df["train_f1"],   label=optimizer_name)
    axes[0, 2].plot(history_df["step"], history_df["train_em"],   label=optimizer_name)

    # Bottom row: dev metrics
    axes[1, 0].plot(history_df["step"], history_df["dev_loss"], label=optimizer_name)
    axes[1, 1].plot(history_df["step"], history_df["dev_f1"],   label=optimizer_name)
    axes[1, 2].plot(history_df["step"], history_df["dev_em"],   label=optimizer_name)

# Titles
axes[0, 0].set_title("Train Loss")
axes[0, 1].set_title("Train F1")
axes[0, 2].set_title("Train EM")
axes[1, 0].set_title("Dev Loss")
axes[1, 1].set_title("Dev F1")
axes[1, 2].set_title("Dev EM")

# Labels and grid
for r in range(2):
    for c in range(3):
        axes[r, c].set_xlabel("Step")
        axes[r, c].grid(True, alpha=0.3)

axes[0, 0].set_ylabel("Loss")
axes[1, 0].set_ylabel("Loss")
axes[0, 1].set_ylabel("Score")
axes[0, 2].set_ylabel("Score")
axes[1, 1].set_ylabel("Score")
axes[1, 2].set_ylabel("Score")

# Legend (one for whole figure)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=max(1, len(labels)))

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

---
## Section 4 — Evaluate

Loads the saved checkpoint and runs inference on the full dev set.

In [ ]:
from EvaluateTools.evaluate import evaluate

for r in runs:
     metrics = evaluate(
      dev_npz       = "_data/dev.npz",
      word_emb_json = "_data/word_emb.json",
      char_emb_json = "_data/char_emb.json",
      dev_eval_json = "_data/dev_eval.json",
      save_dir      = r["save_dir"],
      log_dir       = r["log_dir"],
      ckpt_name     = "model_best.pt",  # best dev F1/EM from training
      test_num_batches = -1,
     )
     print(f"F1: {metrics['f1']:.4f}  |  EM: {metrics['exact_match']:.4f}  |  Loss: {metrics['loss']:.6f}")
     with open(f"experiments/optimizer_compare/{r['optimizer_name']}/final_metrics.json", "w") as f:
         json.dump(metrics, f, indent=2)
         

